# 3. Cart Composition Diff — 담았지만 안 산 사람들의 카트 vs 구매한 사람들의 카트 (cosmetics_train)

**배경**: 2번 노트북까지는 원본 raw 이벤트 로그(2019-Oct~2020-Feb)를 직접 파싱해서 세션/유저 기준으로 remove 패턴을 봤음. 이번엔 별도로 준비된 전처리/FE 완료 데이터셋(`cosmetics_train.csv`)을 사용한다.

**이 데이터셋이 2번 노트북과 다른 점 (data_info.md 기준)**:
- 한 행 = 한 번의 "카트에 상품을 담은 사건"(user_hash, product_id, t)이고, 그 사건이 최종적으로 어떻게 끝났는지(`outcome`)가 이미 라벨링되어 있음: `purchased` / `explicitly_removed`(장바구니에서 명시적으로 삭제) / `silently_abandoned`(담은 후 추가 행동 없음).
- 세션이 30분 단위로 재정의되고, 동일 로그에 최대 48번 반복되던 중복 이벤트가 이미 통합되어 있음 → 2번 노트북에서 발견했던 "세션 내 봇성 중복 remove" 노이즈가 이 데이터에는 원천적으로 덜 섞여 있을 가능성이 높음.
- product_id 체계는 기존 raw 로그와 동일(확인함: 2번 노트북에서 나온 top-lift 후보 product_id들이 모두 이 데이터셋에도 존재) → 교차검증 가능.

**이번 노트북 목표**: "담았지만 안 산 사람들의 카트에는 어떤 상품이 들어있는가", 그리고 "그 상품 구성이 구매한 사람들의 카트 구성과 다른가"를 상품 단위로 확인한다. 단순 빈도가 아니라 상품별 `abandon_rate`의 baseline 대비 lift로 본다(인기 상품이라 빈도만 높은 경우를 배제하기 위해, 2번 노트북과 동일한 원칙).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

TRAIN_CSV = "C:/Users/user/Downloads/data/cosmetics_train.csv"

USECOLS = ['user_hash', 'product_id', 'outcome', 'price', 'p_prior_price_mean', 'cart_repeat_cnt']
df = pd.read_csv(TRAIN_CSV, usecols=USECOLS)
print("rows:", len(df), " products:", df['product_id'].nunique(), " users:", df['user_hash'].nunique())
print(df['outcome'].value_counts())
print(df['outcome'].value_counts(normalize=True))

rows: 4597697  products: 45461  users: 384380
outcome
silently_abandoned    2594888
purchased             1086378
explicitly_removed     916431
Name: count, dtype: int64
outcome
silently_abandoned    0.564389
purchased             0.236287
explicitly_removed    0.199324
Name: proportion, dtype: float64


## 1. Baseline: 담은 것 중 실제 얼마나 안 사는가

`outcome`이 `purchased`가 아니면 abandon(explicit + silent)으로 묶는다.

In [2]:
df['is_abandoned'] = df['outcome'] != 'purchased'
df['is_explicit'] = df['outcome'] == 'explicitly_removed'
df['is_silent'] = df['outcome'] == 'silently_abandoned'

overall_abandon_rate = df['is_abandoned'].mean()
overall_explicit_rate = df['is_explicit'].mean()
overall_silent_rate = df['is_silent'].mean()
print(f"overall abandon rate (explicit+silent): {overall_abandon_rate:.4f}")
print(f"  of which explicit: {overall_explicit_rate:.4f}  silent: {overall_silent_rate:.4f}")

overall abandon rate (explicit+silent): 0.7637
  of which explicit: 0.1993  silent: 0.5644


## 2. 상품별 abandon lift

상품별로 담긴 횟수 대비 abandon된 비율을 구하고, 전체 baseline 대비 몇 배인지(lift)를 계산한다. 표본이 너무 적은 상품은 lift가 튀므로(예: 2번 중 2번 다 abandon → lift 최대) 최소 카트 횟수 기준을 둔다.

In [3]:
cart_count_by_product = df.groupby('product_id').size()
print(cart_count_by_product.describe())
print("products with >=50 cart events:", (cart_count_by_product >= 50).sum())
print("products with >=100 cart events:", (cart_count_by_product >= 100).sum())
print("products with >=200 cart events:", (cart_count_by_product >= 200).sum())

count    45461.000000
mean       101.134973
std        292.821279
min          1.000000
25%         11.000000
50%         33.000000
75%         90.000000
max      22654.000000
dtype: float64
products with >=50 cart events: 17841
products with >=100 cart events: 10355
products with >=200 cart events: 5190


In [4]:
MIN_CART_COUNT = 100

product_summary = df.groupby('product_id').agg(
    cart_count=('is_abandoned', 'size'),
    abandon_count=('is_abandoned', 'sum'),
    explicit_count=('is_explicit', 'sum'),
    silent_count=('is_silent', 'sum'),
    price_mean=('price', 'mean'),
)
product_summary['abandon_rate'] = product_summary['abandon_count'] / product_summary['cart_count']
product_summary['explicit_rate'] = product_summary['explicit_count'] / product_summary['cart_count']
product_summary['silent_rate'] = product_summary['silent_count'] / product_summary['cart_count']

product_summary['abandon_lift'] = product_summary['abandon_rate'] / overall_abandon_rate
product_summary['explicit_lift'] = product_summary['explicit_rate'] / overall_explicit_rate
product_summary['silent_lift'] = product_summary['silent_rate'] / overall_silent_rate

product_summary_f = product_summary[product_summary['cart_count'] >= MIN_CART_COUNT].copy()
print(f"products passing cart_count>={MIN_CART_COUNT} filter: {len(product_summary_f)} / {len(product_summary)}")
print(product_summary_f[['cart_count', 'abandon_rate', 'abandon_lift']].describe())

products passing cart_count>=100 filter: 10355 / 45461
         cart_count  abandon_rate  abandon_lift
count  10355.000000  10355.000000  10355.000000
mean     344.745534      0.770496      1.008882
std      545.333597      0.069589      0.091119
min      100.000000      0.451327      0.590965
25%      135.000000      0.728383      0.953739
50%      200.000000      0.779043      1.020074
75%      355.000000      0.819081      1.072499
max    22654.000000      0.970588      1.270882


## 3. Top lift 상품 (빈도 1위와 비교)

먼저 순수 빈도(abandon_count) 기준 top, 그다음 lift 기준 top을 나란히 본다 — 2번 노트북에서 확인했던 "인기 상품이라 빈도만 높다"는 함정이 이 데이터에서도 똑같이 나타나는지 확인.

In [5]:
print("=== Top 15 by raw abandon_count (frequency) ===")
top_by_freq = product_summary_f.sort_values('abandon_count', ascending=False).head(15)
print(top_by_freq[['cart_count', 'abandon_count', 'abandon_rate', 'abandon_lift']])

=== Top 15 by raw abandon_count (frequency) ===
            cart_count  abandon_count  abandon_rate  abandon_lift
product_id                                                       
5809910          22654          16122      0.711662      0.931846
5809912          11475           8634      0.752418      0.985211
5854897          11672           7559      0.647618      0.847987
5815662          10235           7321      0.715291      0.936597
5751422           9678           6708      0.693118      0.907564
5751383           8569           6086      0.710235      0.929976
5809911           7521           5877      0.781412      1.023176
5700037           9094           5731      0.630196      0.825174
5802432           8957           5684      0.634587      0.830924
5849033           7873           5466      0.694272      0.909074
5816170           6810           5247      0.770485      1.008867
5304              7767           5096      0.656109      0.859105
5792800           7305      

In [6]:
print(f"=== Top 20 by abandon_lift (min cart_count={MIN_CART_COUNT}) ===")
top_by_lift = product_summary_f.sort_values('abandon_lift', ascending=False).head(20)
print(top_by_lift[['cart_count', 'abandon_count', 'abandon_rate', 'abandon_lift', 'price_mean']])

=== Top 20 by abandon_lift (min cart_count=100) ===
            cart_count  abandon_count  abandon_rate  abandon_lift  price_mean
product_id                                                                   
5816556            102             99      0.970588      1.270882    7.092353
5842716            115            109      0.947826      1.241077   10.319565
5893836            125            118      0.944000      1.236067    3.938800
5780325            105             99      0.942857      1.234571    3.032571
5864793            104             98      0.942308      1.233851    2.411650
5767919            104             98      0.942308      1.233851    2.018835
5859471            188            177      0.941489      1.232780    3.934946
5739950            102             96      0.941176      1.232370    4.426931
5845633            115            108      0.939130      1.229691    2.830087
5879297            114            107      0.938596      1.228992    5.567719
5823600     

## 4. 2번 노트북(raw 로그, user_id 기준) 결과와 교차검증

2번 노트북에서 user_id 기준 재실행 후 나온 top 후보: masura 5826993(1.79x), 5886064(1.75x), 5635128(1.61x). 이 데이터셋에서도 같은 product_id들의 순위/lift를 확인한다. 서로 다른 전처리 파이프라인(원본 로그 직접 파싱 vs 미리 정제된 FE 데이터셋)에서 같은 상품이 튀어나오면 노이즈가 아니라 진짜 상품 문제일 가능성이 높아진다.

In [7]:
prior_candidates = [5826993, 5886064, 5635128, 5809910, 5819249, 5760768]
labels = {
    5826993: 'masura (prev #1 lift, user-based)',
    5886064: 'masura (prev #2 lift, user-based)',
    5635128: 'prev #3 lift, user-based',
    5809910: 'grattol (prev freq #1, lift<1)',
    5819249: 'ingarden (old session-based top3, dropped in user-based)',
    5760768: 'runail (old session-based top3, dropped in user-based)',
}
check = product_summary.loc[product_summary.index.isin(prior_candidates)].copy()
check['note'] = check.index.map(labels)
check['lift_rank_pct'] = product_summary['abandon_lift'].rank(pct=True).loc[check.index]
print(check[['cart_count', 'abandon_rate', 'abandon_lift', 'lift_rank_pct', 'note']])

            cart_count  abandon_rate  abandon_lift  lift_rank_pct                                               note
product_id                                                                                                          
5635128            281      0.797153      1.043787       0.423033                           prev #3 lift, user-based
5760768            195      0.866667      1.134807       0.694485  runail (old session-based top3, dropped in use...
5809910          22654      0.711662      0.931846       0.166890                     grattol (prev freq #1, lift<1)
5819249            234      0.850427      1.113544       0.639526  ingarden (old session-based top3, dropped in u...
5826993            147      0.795918      1.042170       0.419513                  masura (prev #1 lift, user-based)
5886064            271      0.808118      1.058144       0.476144                  masura (prev #2 lift, user-based)


## 5. 카트 구성 비교: 구매 카트 vs 미구매 카트 (Top-N)

"담았지만 안 산 카트"와 "구매까지 간 카트" 각각에서 상품 점유율(share) 상위 목록을 비교한다. 순수 점유율은 인기상품 편향이 있으므로, 옆에 lift도 같이 둔다.

In [8]:
purchased_share = df[~df['is_abandoned']]['product_id'].value_counts(normalize=True).rename('share_purchased')
abandoned_share = df[df['is_abandoned']]['product_id'].value_counts(normalize=True).rename('share_abandoned')

share_cmp = pd.concat([purchased_share, abandoned_share], axis=1).fillna(0)
share_cmp['share_ratio_abandoned_over_purchased'] = (share_cmp['share_abandoned'] + 1e-9) / (share_cmp['share_purchased'] + 1e-9)
share_cmp = share_cmp.join(product_summary_f[['cart_count', 'abandon_lift']], how='inner')

print("=== Top 15 상품 점유율, 구매 카트 기준 ===")
print(share_cmp.sort_values('share_purchased', ascending=False).head(15)[['share_purchased', 'share_abandoned', 'cart_count']])
print()
print("=== Top 15 상품 점유율, 미구매(abandoned) 카트 기준 ===")
print(share_cmp.sort_values('share_abandoned', ascending=False).head(15)[['share_purchased', 'share_abandoned', 'cart_count']])

=== Top 15 상품 점유율, 구매 카트 기준 ===
            share_purchased  share_abandoned  cart_count
product_id                                              
5809910            0.006013         0.004591       22654
5854897            0.003786         0.002153       11672
5700037            0.003096         0.001632        9094
5802432            0.003013         0.001619        8957
5751422            0.002734         0.001910        9678
5815662            0.002682         0.002085       10235
5809912            0.002615         0.002459       11475
5304               0.002459         0.001451        7767
5751383            0.002286         0.001733        8569
5849033            0.002216         0.001557        7873
5686925            0.002095         0.001137        6267
5792800            0.002076         0.001438        7305
5843836            0.001528         0.000694        4097
5528035            0.001527         0.001098        5513
5809911            0.001513         0.001674        7521

## 6. 가격이 담았다 뺀 이유인가? (price 관점)

`abandon_lift` 상위 상품군과 하위 상품군의 평균 가격을 비교해서, "비싸서 뺀다"는 가설이 데이터로 뒷받침되는지 확인한다.

In [9]:
ps = product_summary_f.dropna(subset=['price_mean']).copy()
ps['lift_decile'] = pd.qcut(ps['abandon_lift'], 10, labels=False, duplicates='drop')

price_by_decile = ps.groupby('lift_decile').agg(
    n_products=('price_mean', 'size'),
    avg_abandon_lift=('abandon_lift', 'mean'),
    avg_price=('price_mean', 'mean'),
    median_price=('price_mean', 'median'),
)
print(price_by_decile)

top_decile = ps['lift_decile'].max()
high_lift_price = ps.loc[ps['lift_decile'] == top_decile, 'price_mean']
low_lift_price = ps.loc[ps['lift_decile'] == 0, 'price_mean']
print(f"\nhighest-lift decile mean price: {high_lift_price.mean():.2f} (n={len(high_lift_price)})")
print(f"lowest-lift decile mean price: {low_lift_price.mean():.2f} (n={len(low_lift_price)})")
print(f"price ratio (high-lift / low-lift): {high_lift_price.mean() / low_lift_price.mean():.2f}x")

             n_products  avg_abandon_lift  avg_price  median_price
lift_decile                                                       
0                  1036          0.828344   4.245418      2.200362
1                  1035          0.911855   4.823090      2.859388
2                  1046          0.953761   4.899120      2.847982
3                  1025          0.983898   4.763353      2.995536
4                  1036          1.008846   4.953214      2.999347
5                  1035          1.030670   5.130005      3.474813
6                  1036          1.051729   4.975735      3.088167
7                  1036          1.073062   5.261185      3.797352
8                  1036          1.100100   5.294020      3.950000
9                  1034          1.147012   7.192702      4.255218

highest-lift decile mean price: 7.19 (n=1034)
lowest-lift decile mean price: 4.25 (n=1036)
price ratio (high-lift / low-lift): 1.69x


## 7. Explicit vs Silent — "적극적으로 뺀" 상품과 "그냥 방치된" 상품은 다른가

명시적 제거(explicit)는 상품을 보고 능동적으로 거부한 신호에 가깝고, silent abandon은 가격/배송 등 체크아웃 단계 이탈이거나 단순 변심/딴 데 정신 팔림에 가까울 수 있다. 두 lift의 순위가 얼마나 겹치는지 확인한다.

In [10]:
top_explicit = product_summary_f.sort_values('explicit_lift', ascending=False).head(20)
top_silent = product_summary_f.sort_values('silent_lift', ascending=False).head(20)

overlap = set(top_explicit.index) & set(top_silent.index)
print(f"explicit-lift top20 vs silent-lift top20 overlap: {len(overlap)} / 20")
print()
print("=== Top 15 by explicit_lift ===")
print(top_explicit[['cart_count', 'explicit_rate', 'explicit_lift', 'silent_lift']].head(15))
print()
print("=== Top 15 by silent_lift ===")
print(top_silent[['cart_count', 'silent_rate', 'silent_lift', 'explicit_lift']].head(15))

explicit-lift top20 vs silent-lift top20 overlap: 0 / 20

=== Top 15 by explicit_lift ===
            cart_count  explicit_rate  explicit_lift  silent_lift
product_id                                                       
5811700            101       0.445545       2.235279     0.701714
5813053            133       0.413534       2.074682     0.825965
5879284            105       0.409524       2.054564     0.860603
5674697            157       0.401274       2.013175     0.846415
5859471            188       0.398936       2.001447     0.961311
5800963            118       0.398305       1.998280     0.900930
5877222            113       0.398230       1.997904     0.831035
5767915            131       0.396947       1.991465     0.879152
5864598            226       0.393805       1.975705     0.744795
5739950            102       0.392157       1.967435     0.972769
5872981            126       0.388889       1.951040     0.970287
5825559            116       0.387931       1.946234

## 요약

**Baseline**: 전체 카트 추가 사건의 76.37%가 결국 구매로 이어지지 않음(explicit 19.93% + silent 56.44%). 2번 노트북(원본 로그, user 기준)의 "제거율 54.14%"와는 정의가 달라(이쪽은 explicit+silent를 합친 "구매 안 함" 비율, 그쪽은 remove_from_cart 이벤트 기준) 직접 비교는 불가하지만, 방향은 일관됨 — cosmetics는 담아도 대부분 안 산다.

**결정적 발견 — 개별 상품 lift가 이 데이터에서는 훨씬 약하게 나옴**: `cart_count>=100` 필터 기준 abandon_lift 범위가 0.59배~1.27배로, 2번 노트북(원본 로그)에서 나왔던 1.79배~2.28배보다 훨씬 좁음. 25~75% 구간이 0.95~1.07배로 대부분 상품이 baseline 근처에 몰려있음.

**이전 후보 상품들의 재검증 결과 — 전부 기각**: masura 5826993(이전 1.79배) → 이 데이터에서 lift 1.04배(하위 42%ile, 완전히 평범), 5886064(이전 1.75배) → 1.06배(48%ile), 5635128(이전 1.61배) → 1.04배(42%ile). ingarden 5819249, runail 5760768도 각각 1.11배/1.13배로 상위권이 아님(64~69%ile). **즉 원본 로그 기반 분석에서 나온 "문제 상품" 후보들은 이 독립적이고 더 깨끗한(중복 이벤트 사전 제거된) 데이터셋에서는 재현되지 않음** — 특정 상품이 문제라기보다는 세션 분할 방식이나 잔여 노이즈에 좌우된 결과였을 가능성이 높다는 뜻. 상품 단위로 "이 상품이 범인이다"라고 지목하는 건 이 데이터로는 근거가 약함.

**카트 구성(점유율) 비교**: 구매 카트 top15와 미구매 카트 top15는 상품 구성이 거의 동일함(둘 다 5809910, 5809912, 5854897 등 원래 인기 상품이 상위). 즉 "안 사는 사람의 카트에 특별히 다른 상품이 들어있다"는 가설은 점유율만으로는 지지되지 않음 — 다들 비슷한 인기 상품을 담고, 다만 그중 일부를 안 사는 것.

**반전 — 가격이 진짜 신호**: abandon_lift 10분위별 평균가를 보면 최저 lift 분위 평균가 4.25 → 최고 lift 분위 평균가 7.19로 **1.69배** 단조 증가. 개별 상품보다 "가격대"가 이탈과 훨씬 뚜렷하게 연관됨 — 비쌀수록 담고 나서 안 살 확률이 체계적으로 올라감(sticker shock 가설과 일치).

**가장 흥미로운 발견 — explicit vs silent는 완전히 다른 상품 집합**: explicit_lift top20과 silent_lift top20의 겹침이 **0/20**. explicit_lift 상위 상품(예: 5811700, 44.5%가 명시적으로 삭제, lift 2.24배)은 대부분 silent_lift가 오히려 baseline 이하(0.70배)이고, 그 반대도 마찬가지. 게다가 explicit_lift 최대값(2.24배)이 합쳐서 본 abandon_lift 최대값(1.27배)보다 훨씬 큼 — **explicit/silent를 합쳐서 보면 신호가 서로 상쇄되어 희석됨.** "능동적으로 거부당하는 상품"과 "그냥 잊혀지는 상품"은 다른 현상이고, 상품 문제를 찾으려면 explicit_lift만 따로 봐야 함(silent는 가격/체크아웃 이탈 등 상품과 무관한 이유일 가능성이 큼).

**결론**: "담긴 상품 자체가 문제"라는 가설은 순수 상품 점유율이나 합산 abandon_lift로는 약하게만 지지되고, 2번 노트북의 개별 후보 상품들은 이 데이터에서 재현 안 됨. 대신 (1) 가격대가 이탈과 뚜렷한 관계가 있고, (2) explicit_lift 상위 상품들(위 표의 5811700, 5813053, 5879284 등, cart_count>=100 기준)은 "그 상품 자체를 보고 거부"당하는 진짜 상품 문제 후보로 볼 수 있음 — 다만 표본이 n~100~200 수준이라 개별 상품 결론엔 신중해야 함.